In [3]:
import hashlib
import time
import random

#----------------------------------------
# part 1: Proof of Work
#----------------------------------------

#----------------------------------------
# Structure of a block
#----------------------------------------
class Block:
    """Represents a single block in the PoW blockchain."""
    def __init__(self, index, previous_hash, timestamp, data, nonce=0):
        self.index = index                # Position of the block in the chain
        self.previous_hash = previous_hash  # Hash of the previous block
        self.timestamp = timestamp        # Time when block is created
        self.data = data                  # Transaction data (or any info)
        self.nonce = nonce                # Number used once for mining
        self.hash = self.calculate_hash() # Block's own hash

    def calculate_hash(self):
        """Returns the SHA-256 hash of the block's contents."""
        # Concatenate all fields except 'hash' itself (to avoid recursion)
        block_string = f"{self.index}{self.previous_hash}{self.timestamp}{self.data}{self.nonce}"
        # Encode to bytes and compute hash
        return hashlib.sha256(block_string.encode()).hexdigest()

In [4]:
class Blockchain:
    """Manages the chain of blocks using Proof of Work."""
    def __init__(self, difficulty=4):
        self.chain = []                     # List to store blocks
        self.difficulty = difficulty        # Number of leading zeros required in hash
        self.create_genesis_block()         # Initialize the chain with genesis block

    def create_genesis_block(self):
        """Creates and adds the first block (genesis) to the chain."""
        genesis_block = Block(0, "0", time.time(), "Genesis Block")
        genesis_block.hash = genesis_block.calculate_hash()
        self.chain.append(genesis_block)

    def get_last_block(self):
        """Returns the latest block in the chain."""
        return self.chain[-1]

    def proof_of_work(self, block):
        """Mines a block by finding a nonce that makes the block's hash start with 'difficulty' zeros."""
        # Calculate target pattern: e.g., difficulty=4 -> "0000"
        target = "0" * self.difficulty
        # Keep changing nonce until hash meets target
        while block.hash[:self.difficulty] != target:
            block.nonce += 1
            block.hash = block.calculate_hash()
        print(f"Block mined: nonce = {block.nonce}, hash = {block.hash}")

    def add_block(self, data):
        """Creates a new block with given data, mines it using PoW, and adds to chain."""
        last_block = self.get_last_block()
        # Prepare new block (nonce will be adjusted during mining)
        new_block = Block(last_block.index + 1, last_block.hash, time.time(), data)
        # Perform PoW mining to find valid nonce
        self.proof_of_work(new_block)
        # Add mined block to chain
        self.chain.append(new_block)

    def is_chain_valid(self):
        """Verifies that all block hashes are correct and previous_hash links match."""
        for i in range(1, len(self.chain)):
            current = self.chain[i]
            previous = self.chain[i-1]
            # Check that current's stored hash matches recalculation
            if current.hash != current.calculate_hash():
                return False
            # Check that current points to the correct previous hash
            if current.previous_hash != previous.hash:
                return False
        return True


In [5]:
# ----------------------------
# Part 2: Proof of Stake (PoS)
# ----------------------------

class PoSBlock(Block):
    """Inherits from Block and adds a validator field."""
    def __init__(self, index, previous_hash, timestamp, data, validator, nonce=0):
        # Call parent constructor (nonce is not used for mining in PoS but kept for compatibility)
        super().__init__(index, previous_hash, timestamp, data, nonce)
        self.validator = validator          # Who created this block


class PoSBlockchain:
    """Manages the chain using Proof of Stake – validators are chosen based on their stake."""
    def __init__(self):
        self.chain = []
        # Dictionary: validator name -> stake amount (higher stake = higher chance to be chosen)
        self.stakes = {}
        self.create_genesis_block()

    def create_genesis_block(self):
        """Creates genesis block with a default validator."""
        genesis = PoSBlock(0, "0", time.time(), "Genesis Block", validator="Genesis")
        genesis.hash = genesis.calculate_hash()
        self.chain.append(genesis)

    def get_last_block(self):
        return self.chain[-1]

    def select_validator(self):
        """
        Selects a validator randomly based on stake proportions.
        Example: stakes = {'Alice': 30, 'Bob': 70} => Bob has 70% chance.
        """
        if not self.stakes:
            raise Exception("No stakes defined. Add validators first.")
        # Create list of (validator, cumulative weight)
        total_stake = sum(self.stakes.values())
        # Random number between 0 and total_stake
        rand = random.uniform(0, total_stake)
        cumulative = 0
        for validator, stake in self.stakes.items():
            cumulative += stake
            if rand <= cumulative:
                return validator
        # Fallback (should not happen if stakes non-empty)
        return list(self.stakes.keys())[0]

    def add_block(self, data):
        """
        Creates a new block, selects a validator using PoS, adds the block without mining.
        (In PoS no computational work is needed; trust comes from stake.)
        """
        last_block = self.get_last_block()
        validator = self.select_validator()
        # Create new PoS block with nonce=0 (no mining)
        new_block = PoSBlock(last_block.index + 1, last_block.hash, time.time(), data, validator)
        new_block.hash = new_block.calculate_hash()   # Just compute hash once
        self.chain.append(new_block)
        print(f"Block added by validator '{validator}' (stake: {self.stakes[validator]})")

    def is_chain_valid(self):
        """Same verification as PoW, but also checks that validator exists in stakes? Not strictly needed."""
        for i in range(1, len(self.chain)):
            current = self.chain[i]
            previous = self.chain[i-1]
            if current.hash != current.calculate_hash():
                return False
            if current.previous_hash != previous.hash:
                return False
        return True

In [6]:
# ----------------------------
# Part 3: Testing and Comparison
# ----------------------------

def test_pow():
    print("\n=== Proof of Work Simulation ===")
    pow_chain = Blockchain(difficulty=3)  # 3 leading zeros (faster for testing)
    print("Mining block 1...")
    pow_chain.add_block("First transaction: Alice pays Bob 10 BTC")
    print("Mining block 2...")
    pow_chain.add_block("Second transaction: Bob pays Charlie 5 BTC")
    print("\nPoW Blockchain:")
    for block in pow_chain.chain:
        print(f"Index: {block.index}, Hash: {block.hash}, Data: {block.data}, Nonce: {block.nonce}")
    print(f"Chain valid? {pow_chain.is_chain_valid()}")

def test_pos():
    print("\n=== Proof of Stake Simulation ===")
    pos_chain = PoSBlockchain()
    # Define stakes (higher stake = more likely to be chosen as validator)
    pos_chain.stakes = {
        "Alice": 50,
        "Bob": 30,
        "Charlie": 20
    }
    print("Adding blocks (validators chosen by stake weight)...")
    pos_chain.add_block("Alice pays Bob 10 tokens")
    pos_chain.add_block("Bob pays Charlie 5 tokens")
    pos_chain.add_block("Charlie pays Dave 2 tokens")
    print("\nPoS Blockchain:")
    for block in pos_chain.chain:
        print(f"Index: {block.index}, Hash: {block.hash}, Validator: {block.validator}, Data: {block.data}")
    print(f"Chain valid? {pos_chain.is_chain_valid()}")

if __name__ == "__main__":
    test_pow()
    test_pos()


=== Proof of Work Simulation ===
Mining block 1...
Block mined: nonce = 1038, hash = 0004dae90401fac32fc48cc08a8f9e77c2791a297a868b5e0bcd6ba7269aaf0f
Mining block 2...
Block mined: nonce = 1607, hash = 000b829c4fe42b7b5cda84e8de090d52c9d5a131011f3124bab63d1c000e6394

PoW Blockchain:
Index: 0, Hash: d7704b450ef7e60630481961ce23f13f9bfde579c8639f9578cd6be7dd6e8ad5, Data: Genesis Block, Nonce: 0
Index: 1, Hash: 0004dae90401fac32fc48cc08a8f9e77c2791a297a868b5e0bcd6ba7269aaf0f, Data: First transaction: Alice pays Bob 10 BTC, Nonce: 1038
Index: 2, Hash: 000b829c4fe42b7b5cda84e8de090d52c9d5a131011f3124bab63d1c000e6394, Data: Second transaction: Bob pays Charlie 5 BTC, Nonce: 1607
Chain valid? True

=== Proof of Stake Simulation ===
Adding blocks (validators chosen by stake weight)...
Block added by validator 'Bob' (stake: 30)
Block added by validator 'Charlie' (stake: 20)
Block added by validator 'Bob' (stake: 30)

PoS Blockchain:
Index: 0, Hash: 084572fc45d8d86876f7774c4af418978406bce7687a4